# Marker Repo - annotation

In this notebook, clustered h5ad files can be annotated using the marker repo.

## Loading packages

In [ ]:
import markerrepo.marker_repo as mr
import markerrepo.wrappers as wrap
import markerrepo.annotation as annot
import scanpy as sc
from sctoolbox.tools import celltype_annotation

%load_ext autoreload
%autoreload 2

## Settings

Specify path of the cloned repository, the h5ad file which is going to be annotated as well as the organism and the rank genes column of the 'uns' table if available.

In [ ]:
repo_path = "/mnt/workspace/mkessle/projects/annotate_by_marker_and_features"
h5ad_path = "/mnt/workspace/mkessle/master/refdata/hs.h5ad"
organism = mr.select(key="organism")
rank_genes_column = None

Select which cell type annotation methods you want to perform.

In [ ]:
marker_repo = True
SCSA = True

The following cell is for specifying the prefixes of new columns in the 'obs' table that will contain the annotation results. Currently, Marker Repo and SCSA are available.

In [ ]:
mr_obs= "MR"
scsa_obs = "SCSA"

# if you want to compare the annotation with a reference annotation already present in the 'obs' table
reference_obs = "cell_types" 

Load anndata

In [ ]:
adata = sc.read_h5ad(h5ad_path)

In [ ]:
adata.var

## Prepare adata

### Set genes to index, if not already done.

Pick the genes column.

In [ ]:
genes_column = mr.select(whitelist=list(adata.var.columns), heading="genes column")

In [ ]:
adata.var.reset_index(inplace=True)  # remove peaks from index and save them in the column ['index']
adata.var.set_index(genes_column, inplace=True)  # set genes as index
adata.var.index = adata.var.index.astype('str')  # to avoid index being categorical
adata.var_names_make_unique(join='_')

In [ ]:
adata.var

## Prepare annotation

Pick the clustring column you want to annotate.

In [ ]:
column = mr.select(whitelist=list(adata.obs.columns), heading="clustering column")

### Ranking

Pick the rank genes column, if the anndata object already contains one.

In [ ]:
rank_genes_column = mr.select(whitelist=list(adata.uns.keys()), heading="rank genes column")

Rank genes, if not already done.

In [ ]:
rank_genes_column = None

In [ ]:
if not rank_genes_column:
    # adata.uns['log1p']['base'] = None
    rank_genes_column = f'rank_genes_groups_{column}'
    print(f'Ranking genes groups for clusters using obs column {column}')
    sc.tl.rank_genes_groups(adata, groupby=f'{column}', use_raw=False, key_added=rank_genes_column)

In [ ]:
sc.pl.rank_genes_groups_matrixplot(adata, standard_scale='var', n_genes=10, key=rank_genes_column, show=True)

## Create suitable marker list(s)

The paths of the marker lists will be stored in the <b>marker_lists</b> variable. They will work as input for the actual cell type annotation of the next cell. If the index of adata.var contains ensembl IDs, set <b>ensembl=True</b>, otherwise gene symbols are used.

In [ ]:
marker_lists = wrap.create_marker_lists(organism, repo_path=repo_path, style="score", file_name=f"{organism.split(' ')[0]}", ensembl=True)

In [ ]:
marker_lists

## Annotate adata using the created list(s)

In [ ]:
if marker_repo or SCSA:
    for marker_list in marker_lists:
        columns_to_plot = [] if reference_obs is None else [reference_obs]
        
        name = f"{marker_list.split('/')[-1]}"
        annotation_dir = f"./annotation/{name}"

        if marker_repo:
            ct_column = f"{mr_obs}_{name}"
            columns_to_plot.append(ct_column)
            
            # Run Marker Repo annotation
            annot.annot_ct(adata, output_path=annotation_dir, db_path=marker_list,
                           cluster_column=f"{column}", rank_genes_column=rank_genes_column, 
                           ct_column=ct_column)

            # Show scores and alternate cell types of eacht cluster
            print(f"Tables of cell type annotation with clustering {column}")
            annot.show_tables(annotation_dir=annotation_dir, n=5, clustering_column=column)

        if SCSA:
            column_added = f"{scsa_obs}_{name}"
            columns_to_plot.append(column_added)
            
            # Run SCSA annotation
            celltype_annotation.run_scsa(adata, 
                 gene_column=None, 
                 key=rank_genes_column, 
                 column_added=column_added,
                 inplace=True, 
                 species=None, 
                 fc=1.5, 
                 pvalue=0.05, 
                 user_db=annot.reformat_marker_list(marker_list), 
                 celltype_column="cell_name")

        # Plot annotation
        sc.pl.umap(adata, color=columns_to_plot, wspace=0.5)

### Compare the different annotations

In [ ]:
annot.compare_cell_types(adata, column, columns_to_plot)